# Normalization: one fact in one place

Practice notebook for the datafromzero SQL lesson **Normalization: one fact in one place**. The engine is DuckDB and the data is
**cafe-sales**: five tables from a fictional chain of three cafes (synthetic data, CC BY 4.0).

Run the cells from top to bottom. The worked example is written out so you can run it and read the results. The
exercises start empty or half done: write your own query there. The solutions are in the lesson, not here.

Each query sits between `con.sql("""` and `""").show()`: only the text inside the triple quotes is SQL, and `.show()`
prints the result as a table. The cells leave out the closing `;` that the lesson shows.

You change only copies you create here. The cafe-sales tables are never changed, and the setup cell gives you a clean start. Exercises 2 and 3 read the tables you build in steps 5 to 7, so run the worked example first.

## Setup

Run this cell first. It builds an empty database in memory and loads the five cafe-sales tables into it. Run it again at any time to start over.

In [ ]:
%pip install -q duckdb==2.0.0  # the version these lessons were checked with
import duckdb, urllib.request

URL = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/cafe-sales.sql"
script = urllib.request.urlopen(URL).read().decode("utf-8")

con = duckdb.connect()  # an empty database that lives in memory
con.execute(script)     # creates and fills the five cafe-sales tables
print("cafe-sales is loaded")

## Worked example

### Step 1. Build the wide table

In [ ]:
con.sql("""
CREATE TABLE orders_wide AS
SELECT oi.order_item_id, oi.order_id, oi.product_id, oi.quantity, oi.discount_usd,
       o.ordered_at, o.status, o.payment_method,
       o.store_id, s.store_name, s.city AS store_city,
       o.customer_id, c.city AS customer_city, c.loyalty_tier,
       p.product_name, p.category, p.unit_price_usd
FROM order_items AS oi
JOIN orders AS o ON o.order_id = oi.order_id
JOIN stores AS s ON s.store_id = o.store_id
LEFT JOIN customers AS c ON c.customer_id = o.customer_id
JOIN products AS p ON p.product_id = oi.product_id
""").show()

In [ ]:
con.sql("""
DESCRIBE orders_wide
""").show()

### Step 2. Spot the repeated facts

In [ ]:
con.sql("""
SELECT store_id, store_name, store_city, COUNT(*) AS lines
FROM orders_wide
GROUP BY store_id, store_name, store_city
ORDER BY store_id
""").show()

### Step 3. Update anomaly, then undo it

Four cells: `BEGIN`, the `UPDATE`, the `SELECT`, `ROLLBACK`.

In [ ]:
con.sql("""
BEGIN
""").show()

In [ ]:
con.sql("""
UPDATE orders_wide SET store_name = 'Harbor Point' WHERE store_id = 3 AND ordered_at < '2026-05-01'
""").show()

In [ ]:
con.sql("""
SELECT store_name, COUNT(*) AS lines FROM orders_wide WHERE store_id = 3 GROUP BY store_name ORDER BY store_name
""").show()

In [ ]:
con.sql("""
ROLLBACK
""").show()

### Step 4. Insert and delete anomalies

In [ ]:
con.sql("""
SELECT COUNT(DISTINCT product_id) AS sold_products, COUNT(DISTINCT customer_id) AS ordering_customers FROM orders_wide
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS lines FROM orders_wide WHERE customer_id = 42
""").show()

### Step 5. Split out the product facts (2NF)

In [ ]:
con.sql("""
CREATE TABLE n_product (
  product_id INTEGER PRIMARY KEY,
  product_name VARCHAR NOT NULL,
  category VARCHAR NOT NULL,
  unit_price_usd DECIMAL(6, 2) NOT NULL
);
INSERT INTO n_product
SELECT DISTINCT product_id, product_name, category, unit_price_usd FROM orders_wide
""").show()

### Step 6. Split out the store facts (3NF), then the rest

In [ ]:
con.sql("""
CREATE TABLE n_store (store_id INTEGER PRIMARY KEY, store_name VARCHAR NOT NULL, store_city VARCHAR NOT NULL);
INSERT INTO n_store SELECT DISTINCT store_id, store_name, store_city FROM orders_wide
""").show()

Done for you: the same recipe for the customer, order and line tables. Notice `WHERE customer_id IS NOT NULL`: the walk-in order has no customer.

In [ ]:
con.sql("""
CREATE TABLE n_customer (
  customer_id INTEGER PRIMARY KEY,
  customer_city VARCHAR NOT NULL,
  loyalty_tier VARCHAR NOT NULL
);
INSERT INTO n_customer
SELECT DISTINCT customer_id, customer_city, loyalty_tier FROM orders_wide WHERE customer_id IS NOT NULL;
CREATE TABLE n_order (
  order_id INTEGER PRIMARY KEY,
  ordered_at TIMESTAMP NOT NULL,
  status VARCHAR NOT NULL,
  payment_method VARCHAR NOT NULL,
  store_id INTEGER NOT NULL REFERENCES n_store (store_id),
  customer_id INTEGER REFERENCES n_customer (customer_id)
);
INSERT INTO n_order
SELECT DISTINCT order_id, ordered_at, status, payment_method, store_id, customer_id FROM orders_wide;
CREATE TABLE n_line (
  order_item_id INTEGER PRIMARY KEY,
  order_id INTEGER NOT NULL REFERENCES n_order (order_id),
  product_id INTEGER NOT NULL REFERENCES n_product (product_id),
  quantity INTEGER NOT NULL,
  discount_usd DECIMAL(6, 2) NOT NULL
);
INSERT INTO n_line
SELECT order_item_id, order_id, product_id, quantity, discount_usd FROM orders_wide;
SELECT (SELECT COUNT(*) FROM n_store) AS stores, (SELECT COUNT(*) FROM n_customer) AS customers,
       (SELECT COUNT(*) FROM n_order) AS orders, (SELECT COUNT(*) FROM n_line) AS lines,
       (SELECT COUNT(*) FROM n_product) AS products
""").show()

### Step 7. Prove the answers did not change

In [ ]:
con.sql("""
CREATE VIEW orders_rebuilt AS
SELECT l.order_item_id, l.order_id, l.product_id, l.quantity, l.discount_usd,
       o.ordered_at, o.status, o.payment_method,
       o.store_id, s.store_name, s.store_city,
       o.customer_id, c.customer_city, c.loyalty_tier,
       p.product_name, p.category, p.unit_price_usd
FROM n_line AS l
JOIN n_order AS o ON o.order_id = l.order_id
JOIN n_store AS s ON s.store_id = o.store_id
LEFT JOIN n_customer AS c ON c.customer_id = o.customer_id
JOIN n_product AS p ON p.product_id = l.product_id
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS only_in_rebuilt FROM (SELECT * FROM orders_rebuilt EXCEPT SELECT * FROM orders_wide)
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS only_in_wide FROM (SELECT * FROM orders_wide EXCEPT SELECT * FROM orders_rebuilt)
""").show()

In [ ]:
con.sql("""
SELECT store_name, SUM(quantity * unit_price_usd - discount_usd) AS revenue
FROM orders_wide WHERE status = 'completed' GROUP BY store_name ORDER BY store_name
""").show()

In [ ]:
con.sql("""
SELECT store_name, SUM(quantity * unit_price_usd - discount_usd) AS revenue
FROM orders_rebuilt WHERE status = 'completed' GROUP BY store_name ORDER BY store_name
""").show()

## Exercise 1. Split out the customer facts

Done for you: the empty table. Write the `INSERT ... SELECT DISTINCT` from `orders_wide` and check that the new table has one row per customer.

In [ ]:
con.sql("""
CREATE TABLE n_customer_ex1 (
  customer_id INTEGER PRIMARY KEY,
  customer_city VARCHAR NOT NULL,
  loyalty_tier VARCHAR NOT NULL
)
""").show()

In [ ]:
# INSERT INTO n_customer_ex1 ... SELECT DISTINCT ... FROM orders_wide ...

In [ ]:
# Check: COUNT(*) and COUNT(DISTINCT customer_id) of n_customer_ex1

## Exercise 2. Did splitting change the units sold per category?

Before you run anything, write one sentence in the next cell: how will the two result tables compare? Then write units per category (the sum of `quantity`) from `orders_wide`, the same from `orders_rebuilt`, and a query that compares them with `EXCEPT`.

In [ ]:
# Expected result, in words:
#

In [ ]:
# Units per category from orders_wide

In [ ]:
# Units per category from orders_rebuilt

In [ ]:
# The comparison with EXCEPT

## Exercise 3. Spot the flaw

A colleague rebuilds the wide table with `JOIN` instead of `LEFT JOIN` for the customers. The check does not return 0, and the colleague says the split lost a line. Run the cells, then find out what went wrong.

In [ ]:
con.sql("""
CREATE VIEW orders_rebuilt_bad AS
SELECT l.order_item_id, l.order_id, l.product_id, l.quantity, l.discount_usd,
       o.ordered_at, o.status, o.payment_method,
       o.store_id, s.store_name, s.store_city,
       o.customer_id, c.customer_city, c.loyalty_tier,
       p.product_name, p.category, p.unit_price_usd
FROM n_line AS l
JOIN n_order AS o ON o.order_id = l.order_id
JOIN n_store AS s ON s.store_id = o.store_id
JOIN n_customer AS c ON c.customer_id = o.customer_id
JOIN n_product AS p ON p.product_id = l.product_id
""").show()

In [ ]:
con.sql("""
SELECT COUNT(*) AS only_in_wide FROM (SELECT * FROM orders_wide EXCEPT SELECT * FROM orders_rebuilt_bad)
""").show()

In [ ]:
# Which line is it? What is the right join?